# Namespaces, Scope & Decorators — Session 12

Welcome to Session 12, the session where Python's inner machinery becomes visible. So far we have used functions as black boxes: we call them, they run, they return. Today we open the box and look at what is actually happening with the **names** we use — where they live, how Python finds them, and how functions can decorate, wrap, and transform other functions.

The first half of the session is about **namespaces and scope**. Every name in your program — a variable, a function, a class — lives in a namespace, which is essentially a dictionary mapping identifiers to objects. There are four of them stacked around your code: the **built-in** namespace, the **global** (module) namespace, the **enclosing** namespace of outer functions, and the **local** namespace of the function currently running. The **LEGB rule** is the search order Python follows when you use a name: Local, Enclosing, Global, Built-in. Understanding this rule explains a whole family of behaviours — why a function can read a global, why assigning to a name inside a function creates a brand new local, why `UnboundLocalError` appears, and what the `global` and `nonlocal` keywords are for. It also teaches you how *closures* work, because a nested function keeps access to its enclosing function's variables.

The second half is about **decorators**, one of the most beloved features of Python. The secret ingredient is that Python functions are **first-class citizens**: they can be passed as arguments, stored in variables, and returned from other functions. A decorator simply exploits that — it is a function that receives another function, wraps some extra behaviour around it, and hands back a new function. The `@decorator` syntax is just sugar for `func = decorator(func)`. We build our first wrapper, then a real timing decorator, then face the metadata problem that `functools.wraps` solves, and finally we build parameterised decorators with three nested layers.

By the end, decorators will feel less like magic and more like a natural consequence of the namespace and closure machinery you just studied. Along the way, these ideas power frameworks such as Flask routes and Django's ORM — so this is the session that makes professional Python frameworks understandable.

# 1. Namespaces and the LEGB Rule

## 1.1 A namespace is a dictionary of names

A **namespace** is exactly what it sounds like: a space that holds names. Programmatically, it is a dictionary whose keys are identifiers (like `a`, `temp`, `print`) and whose values are the objects they refer to. Python maintains four of them around every piece of code:

| Namespace | What it holds |
|---|---|
| Built-in | `print`, `len`, `dict`, `max` … |
| Global | everything defined at the top of a module |
| Enclosing | variables of outer functions around a nested function |
| Local | names created inside the currently executing function |

A **scope** is the textual region of the program where a given namespace is directly accessible — where a name can be used without a qualifier. The two concepts go together: the namespace is the dictionary, the scope is the territory.

When Python meets a name in your code, it does not guess — it searches from the inside out, strictly in this order: **L**ocal, **E**nclosing, **G**lobal, **B**uilt-in. The first hit wins. If no scope in the chain contains the name, Python raises `NameError`. Memorise the acronym LEGB; it is tested in interviews and it silently explains most of the surprising behaviour you will see in the code cells that follow — for instance, why a variable named `a` inside a function and a variable named `a` at the top level are two completely different objects living in different namespaces.

## Local vs Global scope

```python
a = 2              # global

def temp():
    b = 3          # local — sirf is function ke andar
    print(b)

temp()
print(a)           # global accessible
# print(b)         # NameError! `b` local hi tha
```

- **Global** — file top-level, visible everywhere
- **Local** — created inside a function, accessible only there
- `b` created inside `temp()` won't be found outside
- Namespace = a dictionary of names, scope = a region


## Deep Dive : Namespace & local/global scope

```python
a = 2                     # global var — module namespace

def temp():
    b = 5                 # local var — function namespace
    print(b)
temp()
print(a)                  # ✓ global outside
# print(b)                # NameError — b local scope me
```

- **Namespace** — a mapping of names (variables, functions) to objects
- Every function has its OWN namespace
- Module namespace, function namespace, builtin namespace
- Global: module level, accessible everywhere
- Local: inside a function, not visible outside the function
- Scope = the region where a name is visible
- LEGB lookup order: Local → Enclosing → Global → Builtin
- Strategy: read globals freely inside; writing needs the keyword
- Namespaces are daisy-chained (lexically)


In [47]:
# local and global
# global var
a = 2

def temp():
  # local var
  b = 3
  print(b)

temp()
print(a)

3
2


## Same name local vs global

```python
a = 2          # global

def temp():
    a = 3      # LOCAL nayi `a` — global ko nahi chhoTa
    print(a)

temp()         # 3
print(a)       # 2 — global untouched
```

- An assignment inside a function creates a **local**
- The global `a` stays as it is — this is shadowing
- The two namespaces are separate; `a` is its own thing in each place


## Deep Dive : Same-name shadowing

```python
a = 2                         # global

def temp():
    a = 5                     # LOCAL a — separate!
    print(a)                  # 5 — local wins
temp()
print(a)                      # 2 — global untouched
```

- An assignment `a = 5` in a function → a **new local** variable
- Without the assignment, the global a is only READ
- An assignment anywhere in the function → the whole function treats it as local (shadowing)
- The same name is allowed — the scope differentiates
- Shadowing hides the global inside the function scope
- To prevent this: you need `global a`
- No local + global exists → reads the global (pass-through)
- This is called "LEGB": the function scope comes first
- Beginners get confused here — take 10 minutes to conceptualize it


In [9]:
# local and global -> same name
a = 2

def temp():
  # local var
  a = 3
  print(b)

temp()
print(a)

In [48]:
# local and global -> local does not have but global has
a = 2

def temp():
  # local var
  print(a)

temp()
print(a)


2
2


## Editing a global — the global keyword

```python
a = 2

def temp():
    global a     # ab mai global `a` ko badluga
    a += 1
    print(a)

temp()           # 3
print(a)         # 3 — global change ho gaya
```

- Without declaring it, `a += 1` gives `UnboundLocalError` (reading is also local)
- `global a` — tell the function "use this global one"
- Reading a global (without `global`) is allowed, modifying is not
- Use `global` sparingly — to avoid bad practice


## Deep Dive : Editing globals — the global keyword

```python
a = 2
def temp():
    global a        # a = global wala, not local
    a += 1          # modifies GLOBAL a
temp()
print(a)            # 3 — global changed
```

- Without `global`: `a += 1` inside → UnboundLocalError
- `global a` — tells the function "use this global"
- When needed: counters, config, shared state updates
- Risky — it hurts readability (hidden dependency)
- Modern code: prefer return values / class-state design
- Common error: `UnboundLocalError: local variable referenced before assignment`
- Calling a method on a global list or dict (append) — no global needed (mutation is fine)
- Rebinding (a = x) needs global; mutating (L.append) does not
- global must be at the top of the function


In [49]:
# local and global -> editing global
a = 2

def temp():
  # local var
  a += 1
  print(a)

temp()
print(a)

UnboundLocalError: ignored

In [50]:
a = 2

def temp():
  # local var
  global a
  a += 1
  print(a)

temp()
print(a)

3
3


## Creating a global from a local

```python
def temp():
    global a
    a = 1         # global me naya name ban gaya!
    print(a)

temp()
print(a)          # 1 — ab global me hai
```

- With `global a`, an assignment creates a new entry in the global namespace
- You can create a global from inside a function too
- Rarely needed — usually compared against module level


## Deep Dive : Creating a global from a function

```python
def temp():
    global c
    c = 100        # global namespace me naya banaya
temp()
print(c)           # 100 — global bana
```

- `global c` + assignment → CREATE in the global namespace
- Making a variable global from within a function = sloppy design
- Prefer returning and assigning at the module level:
```python
c = temp()   # function returns value
```
- Global-from-inside → maintenance pain
- A shared state's side effects become unclear
- The global registry is an anti-pattern (global styling at its worst)
- A rare real need — module-level config via constants is fine
- Encapsulation via classes with attributes
- namespaces live as long as the module imports are alive


In [ ]:
# local and global -> global created inside local
def temp():
  # local var
  global a
  a = 1
  print(a)

temp()
print(a)

## Parameters are local

```python
def temp(z):
    print(z)      # z function-ka-local hai

a = 5
temp(5)           # z = 5
# print(z)        # NameError! z sirf function ke andar
```

- Function parameters belong to the **local scope** too
- After the call, `z` vanishes outside
- This is why every function handles its own parameters


## Deep Dive : Parameters are local

```python
def temp(z):       # z is LOCAL to temp
    print(z)
temp(10)           # prints 10
z                  # NameError outside
```

- Parameters = the function's local namespace
- Values are passed copy-in (for immutables)
- Outside the scope z is undefined
- Every call creates a fresh local frame
- A mutable param -> the reference is shared (mutation affects the caller)
- Default param trap: `def f(L=[])` — shared default! use None
- Function local variables vanish after return (except closures)
- Call arguments are evaluated at call time
- Scope rule: parameters behave like locals


In [51]:
# local and global -> function parameter is local
def temp(z):
  # local var
  print(z)

a = 5
temp(5)
print(a)
print(z)

5
5


NameError: ignored

# 2. Built-in Scope

## 2.1 Names that are always available

The **built-in namespace** is the outermost layer of the LEGB chain — the fallback when Local, Enclosing, and Global all come up empty. It contains every built-in Python function and constant: `print`, `len`, `type`, `max`, `min`, `list`, `int`, `input`, and hundreds more. The interpreter populates it at startup, which is why `print` works in every function with no import at all.

The notebook shows two things about it. First, we can inspect the whole built-in namespace directly:

```python
import builtins
print(dir(builtins))
```

`dir()` returns every name in the module, so you can browse the full built-in catalogue. Second, the namespace is only a *fallback*, not a fortress — we can shadow a built-in by defining our own name in an inner scope:

```python
def max():      # shadows the built-in max
    ...
```

After that, `max` resolves to our function (Local/Global beats Built-in), which usually breaks code expecting the real `max`. The lesson: treat built-in names as reserved — there is almost never a good reason to redefine `max`, `list`, or `len`. The lookup order also guarantees that, unless we shadow it, a built-in is reachable from anywhere in the program.

In [53]:
# built-in scope
import builtins
print(dir(builtins))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EnvironmentError', 'Exception', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning', 'StopAsyncIteration', 'StopIteration', 'SyntaxError', 'SyntaxWarning', 'SystemError

## Built-in scope

```python
import builtins
print(dir(builtins))   # sab built-in names ki list
```

- **Built-in** = the outermost scope — `print`, `len`, `type`...
- With `dir(builtins)` you can see all the built-in names
- Built-ins are always available from inside a function
- If you override one, you shadow it — see the next cell


In [ ]:
# how to see all the built-ins

In [68]:
# renaming built-ins
L = [1,2,3]
print(max(L))
def max():
  print('hello')

print(max(L))

TypeError: ignored

# 3. Enclosing Scope and nonlocal

## 3.1 Nested functions read the outer scope

So far we examined Local and Global. The **Enclosing** scope sits between them and only exists when a function is defined *inside* another function:

```python
def outer():
    a = 1
    def inner():
        print(a)      # finds `a` in the enclosing scope
    inner()
    print('outer function')

outer()
```

When `inner` looks for `a`, its own Local namespace has no `a`, so the search moves to the Enclosing scope — `outer`'s locals — and finds it. This is a **closure** in action: the inner function carries a reference to its enclosing environment, even after `outer` returns. Closures are the foundation of decorators.

Reading an enclosing variable is free, but *reassigning* one is not. Because `a += 1` inside `inner` counts as an assignment, Python would treat `a` as local to `inner` and complain that it was used before assignment. The `nonlocal` keyword fixes this — it tells the function "the `a` I mean belongs to the enclosing scope":

```python
def outer():
    a = 1
    def inner():
        nonlocal a
        a += 1
    inner()
    print(a)      # 2
```

Compare the two keywords: `global` points at the module-level namespace, `nonlocal` points at the nearest enclosing *function* namespace. When a closure needs to keep running state, `nonlocal` is how it mutates that state.

## Deep Dive : Enclosing / nonlocal scope

```python
def outer():
    a = 10                  # enclosing scope (outer ka local)
    def inner():
        print(a)            # closure — outer ka a read
    inner()
```

- **Enclosing scope** — the outer function's local within a nested function
- inner reads the outer's variables (a closure)
- LEGB: inner first its own local, then enclosing (outer), then global
- To WRITE the outer's variable → you need `nonlocal`

```python
def outer():
    a = 1
    def inner():
        nonlocal a   # outer ka a modify
        a += 1
    inner()
    print(a)          # 2
```
- nonlocal vs global — nonlocal = the enclosing function
- Closures capture the enclosing environment
- Loop/closure bug: late binding (functions capture the variable, not the value)
- Foundation for decorators! (the wrapper reads the function)
- Interview: closures + nonlocal


In [65]:
# Enclosing scope
def outer():
  def inner():
    print(a)
  inner()
  print('outer function')


outer()
print('main program')

1
outer function
main program


## The nonlocal keyword

```python
def outer():
    a = 1
    def inner():
        nonlocal a     # outer ka `a` modify
        a += 1
        print('inner', a)
    inner()

outer()   # inner 2
```

- `nonlocal` — to change an enclosing (outer) variable
- Reading is allowed; declare `nonlocal` to modify
- This is core to closures (decorators are built on it)


In [67]:
# nonlocal keyword
def outer():
  a = 1
  def inner():
    nonlocal a
    a += 1
    print('inner',a)
  inner()
  print('outer',a)


outer()
print('main program')

inner 2
outer 2
main program


In [16]:
# Summary

# 4. Decorators

## 4.1 Functions that decorate functions

A **decorator** is a function that receives another function as input, adds some functionality (the "decoration"), and returns the enhanced version. It is possible only because Python functions are **first-class citizens** — they are ordinary objects that can be passed around like any value:

```python
def modify(func, num):
    return func(num)

def square(num):
    return num ** 2

modify(square, 4)   # 16 — we passed the function itself
```

Notice there is no `()` after `square` in `modify(square, 4)` — we hand over the function object, and `modify` calls it through its parameter. Any function that takes or returns another function is a **higher-order function**, and a decorator is exactly that: a higher-order function applied at definition time.

The first decorator in the notebook wraps its target with banner lines:

```python
def my_decorator(func):
    def wrapper():
        print('***********************')
        func()
        print('***********************')
    return wrapper

@my_decorator
def hello():
    print('hello')
```

`@my_decorator` is pure sugar for `hello = my_decorator(hello)`. From then on, the name `hello` refers to `wrapper`, which runs the original body in between its own statements. The wrapper is a *closure* — it captured `func` — and that captured reference is what lets the decoration "wrap" the original.

Python ships a small set of built-in decorators (`@staticmethod`, `@classmethod`, `@property`, `@abstractmethod`), but users can write their own for logging, timing, validation, and caching.

## Functions are first-class citizens

```python
def modify(func, num):
    return func(num)

def square(num):
    return num ** 2

modify(square, 4)     # 16 — function as value pass
```

- In Python functions are also values — pass, return, store, all fine
- `func` as an argument — a higher-order function
- Decorators exist because of this power


## Deep Dive : Functions are first-class

```python
def modify(func, num):
    return func(num)

def square(x):
    return x**2

result = modify(square, 3)    # function passed as arg
print(result)                 # 9
```

- Functions are objects: pass as args, store, return
- Already seen in Session 6 — recap: a `func` reference is passed
- HOF: a function that takes/returns functions
- modify(square, 3): square → modify's parameter
- Decorators USE this — function in, function out
- map/filter/reduce are built on it
- Store: `L = [square, cube]`
- identity fn == fn
- This sets the stage — decorators = a function wrapping a function


In [71]:
# Python are 1st class function

def modify(func,num):
  return func(num)

def square(num):
  return num**2

modify(square,2)

4

## First decorator

```python
def my_decorator(func):
    def wrapper():
        print('***********************')
        func()
        print('***********************')
    return wrapper

@my_decorator
def hello():
    print('hello')
```

- A decorator = a function that takes a function and returns a wrapper
- `@my_decorator` == `hello = my_decorator(hello)`
- When you call `hello()`, the wrapper runs — around the original
- Pre/post processing — logging, timing, checks


## Deep Dive : First decorator — closure + wrapper

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print('before call')
        result = func(*args, **kwargs)
        print('after call')
        return result
    return wrapper          # returns the wrapped function

@my_decorator
def hello():
    print('hi')
```

- `@decorator` — syntactic sugar = `func = decorator(func)`
- The decorator receives `func`, defines `wrapper`, returns wrapper
- wrapper extends the behavior (before/after)
- Closure: the wrapper holds the func reference
- Calling hello() → the actual wrapper runs → func() runs
- Return value passthrough: `result = func(...)` + `return result`
- **args/**kwargs forwarding — preserves the signature
- Real usage: logging, timing, auth, caching
- Decorators = HOFs applied at definition time
- Behavior injection without touching the original


In [73]:
# simple example

def my_decorator(func):
  def wrapper():
    print('***********************')
    func()
    print('***********************')
  return wrapper

def hello():
  print('hello')

def display():
  print('hello nitish')

a = my_decorator(hello)
a()

b = my_decorator(display)
b()

***********************
hello
***********************
***********************
hello nitish
***********************


In [ ]:
# more functions

In [ ]:
# how this works -> closure?

In [22]:
# python tutor

In [74]:
# Better syntax?
# simple example

def my_decorator(func):
  def wrapper():
    print('***********************')
    func()
    print('***********************')
  return wrapper

@my_decorator
def hello():
  print('hello')

hello()

***********************
hello
***********************


## Real decorator — timer

```python
import time

def timer(func):
    def wrapper(*args):
        start = time.time()
        func(*args)
        end = time.time()
        print('took', end - start)
    return wrapper

@timer
def do_something():
    time.sleep(2)
```

- `*args`/`**kwargs` — pass every kind of argument through the wrapper
- Timing, logging, access checks — what decorators are perfect for
- Original function unchanged, yet the call is enriched


## Deep Dive : Real decorator — timer

```python
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f'{func.__name__} took {end-start:.4f}s')
        return result
    return wrapper

@timer
def slow_task():
    time.sleep(1)
slow_task()     # slow_task took 1.0004s
```

- The decorator's real power: cross-cutting concerns
- A timer measures any function without editing its code
- `func.__name__` — introspection is preserved (mostly)
- siMetrics: time, log, retry, cache
- Code simplification: wrap + annotation
- A decorator is a HOF returning a wrapper
- Function composition with `@a @b` — a(b(f))
- The "extra capability" is readable right at the definition
- Frameworks (Flask routes, Django ORM) use them heavily


In [81]:
# anything meaningful?
import time

def timer(func):
  def wrapper(*args):
    start = time.time()
    func(*args)
    print('time taken by',func.__name__,time.time()-start,'secs')
  return wrapper

@timer
def hello():
  print('hello wolrd')
  time.sleep(2)

@timer
def square(num):
  time.sleep(1)
  print(num**2)

@timer
def power(a,b):
  print(a**b)

hello()
square(2)
power(2,3)


hello wolrd
time taken by hello 2.003671884536743 secs
4
time taken by square 1.0009939670562744 secs
8
time taken by power 2.1696090698242188e-05 secs


# 5. Wrapping Without Losing Identity

## 5.1 The metadata problem and functools.wraps

There is a subtle bug hiding inside every plain decorator. When `@my_decorator` reassigns `hello = my_decorator(hello)`, the object now bound to `hello` is the `wrapper` function — and the wrapper carries its **own** metadata, not the original's:

```python
@my_decorator
def greet():
    '''Print a greeting.'''
    print('hi')

greet.__name__   # 'wrapper' — the original name lost!
greet.__doc__    # None — the docstring lost!
```

The name, docstring, and even the signature that debugging tools, `help()`, and documentation generators rely on have been replaced by the wrapper's. The function behaves correctly but *identifies* incorrectly.

The fix lives in the standard library:

```python
from functools import wraps

def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        ...
    return wrapper
```

`@wraps(func)` copies `__name__`, `__doc__`, `__module__`, and the stored signature from the original function onto the wrapper, so an outer decorator no longer masquerades as a stranger. This is the universal professional convention — a decorator without `@wraps` breaks introspection, which is worse than it sounds: tracebacks, `help()`, and IDE tooltips all report the wrapper instead of the wrapped function.

## Deep Dive : Problem with a plain wrapper

```python
def my_decorator(func):
    def wrapper(*args, **kwargs):
        ...
    return wrapper

@my_decorator
def greet():
    '''docstring'''
    ...

print(greet.__name__)   # ❌ 'wrapper' — name lost!
print(greet.__doc__)    # ❌ None — doc lost!
```

- A decorated function loses its metadata: `__name__`, `__doc__`, `__module__`
- Introspection tools see the wrapper, not the original
- Fix: `functools.wraps`:

```python
from functools import wraps
def my_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        ...
    return wrapper
```
- `@wraps(func)` copies __name__, __doc__, signature-ish
- Best practice: ALWAYS use @wraps
- debug & trace stay clean
- othera — functools.wraps is essential for professional decorators
- Without it — the API/documentation is broken


In [ ]:
# A big problem

# 6. Decorators That Take Arguments

## 6.1 The three-layer onion

So far our decorators took only the function. What if the decoration should depend on a *parameter* — like a type we want to validate against? Look at the syntax the notebook uses:

```python
@sanity_check(int)
def square(num):
    print(num ** 2)
```

Note the extra parentheses: `@name(args)` means `name(args)` runs *first*, returning a function, and *that* function receives `square`. We therefore need **three** nested layers instead of two:

```python
def sanity_check(data_type):          # layer 1: takes OUR argument
    def outer_wrapper(func):          # layer 2: takes the target function
        def inner_wrapper(*args):     # layer 3: the real wrapper logic
            if type(*args) == data_type:
                func(*args)
            else:
                raise TypeError('datatype check failed')
        return inner_wrapper
    return outer_wrapper
```

Layer 1 captures `data_type`, layer 2 captures `func`, layer 3 performs the runtime check and calls the original. The full evaluation is `square = sanity_check(int)(square)`.

This gives us *parameterised* decorations: type validation, role-based access control, rate limits. Flask's `@app.route('/path')` is precisely this pattern. The three layers look intimidating, but the mental model is an onion — each layer closes over the previous one’s variables: arguments, then function, then the check-and-call wrapper.

In [46]:
# One last example -> decorators with arguments


## Deep Dive : Decorators with arguments

```python
@checkdt(int)       # decorator that TAKES args
def square(num):
    print(num ** 2)
```
Structure: 3 layers!
```python
def sanity_check(data_type):
    def outer_wrapper(func):
        def inner_wrapper(*args, **kwargs):
            for arg in args:
                if not isinstance(arg, data_type):
                    raise TypeError(f'expected {data_type}')
            return func(*args, **kwargs)
        return inner_wrapper
    return outer_wrapper
```

- `@deco(arg)` = call deco first → returns the actual decorator
- Layer 1: takes OUR argument (data_type)
- Layer 2: takes the target function
- Layer 3: the wrapper does the runtime work — the arg check
- Usage: type validation, auth roles, rate limits
- Real parameterized decorators everywhere (flask route!)
- The extra () pair = the factory call
- Execution: `sanity_check(int)(square)`
- Powerful but the 3 layers are confusing — draw the onion diagram


In [82]:
@checkdt(int)
def square(num):
  print(num**2)

## The structure of sanity_check

```python
def sanity_check(data_type):          # layer 1: args
    def outer_wrapper(func):          # layer 2: function
        def inner_wrapper(*args):     # layer 3: real logic
            if type(*args) == data_type:
                print('number is integer')
            else:
                print('number is not integer')
        return inner_wrapper
    return outer_wrapper

@checkdt(int)   # == square = sanity_check(int)(square)
```

- 3 layers = args + func + the args call
- `@name(args)` — the first call is with the arguments
- Then you get the decorator function — and the original function goes onto it
- Remember the layer counting: args → func → wrapper logic


In [95]:
def sanity_check(data_type):
  def outer_wrapper(func):
    def inner_wrapper(*args):
      if type(*args) == data_type:
        func(*args)
      else:
        raise TypeError('Ye datatype nai chalega')
    return inner_wrapper
  return outer_wrapper

@sanity_check(int)
def square(num):
  print(num**2)

@sanity_check(str)
def greet(name):
  print('hello',name)

square(2)

4


## Summary

This session connected Python's naming machinery to one of its most expressive features. You learned that every name lives in a namespace — a dictionary of identifiers — and that Python finds it by scanning the LEGB chain: Local, Enclosing, Global, Built-in. That one rule explains shadowing, `UnboundLocalError`, the `global` keyword for module-level reassignment, and the `nonlocal` keyword for editing enclosing-function variables inside closures. You then met closures as the bridge to decorators: because functions are first-class, a higher-order function can receive a function, wrap extra behaviour around it in a closure, and return the enhanced version — the `@decorator` syntax being nothing more than `func = decorator(func)`. You built banner and timing decorators, discovered how a plain wrapper destroys `__name__` and `__doc__` (fixed by `functools.wraps`), and unpacked parameterised decorators into their three-layer onion. Namespaces explain *where* names live; decorators show how that machinery lets you compose behaviour.